# Gluten/Velox — 02: Velox Performance Deep Dive

## What you will learn
In notebook 01 we learned *which* operators run on Velox.
In this notebook we measure *how much faster* they are.

You will:
1. Understand why Velox is faster than JVM Spark
2. Benchmark every major operator category: scan, filter, aggregation, join, sort
3. Measure the effect of data types on performance
4. Compare Parquet reading speed (Velox native reader vs JVM)
5. Profile CPU and memory efficiency
6. Build a comprehensive speedup report

## Why is Velox faster than JVM Spark?

```
JVM Spark execution path:
  Parquet file
    → JVM Parquet reader (Java)
    → Deserialize to JVM objects (Row/InternalRow)
    → Process row-by-row with WholeStageCodegen
    → Serialize results back
    → GC pressure on all intermediate objects

Velox execution path:
  Parquet file
    → Native Parquet reader (C++)
    → Arrow columnar batch (stays in native memory)
    → Vectorized SIMD processing (AVX2/AVX-512)
    → Results stay in native memory
    → No GC (not on JVM heap)
```

**Key advantages:**
- **Vectorization**: processes 256-512 bits per CPU instruction (SIMD)
- **Column-at-a-time**: operates on entire columns, not row-by-row
- **Native memory**: no JVM GC on data buffers
- **Native Parquet reader**: avoids JVM deserialization overhead
- **Expression compilation**: Velox compiles expressions to native code

**Typical speedups with Gluten 1.6.0 + Spark 4.0.2:**
| Operation | Typical speedup |
|---|---|
| Aggregation (SUM, AVG, COUNT) | 2–4x |
| Filter + scan (Parquet) | 3–6x |
| Hash join | 2–3x |
| Sort | 1.5–2.5x |
| String operations | 1.5–3x |


In [ ]:
import os, time, random, datetime, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'
GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'

spark = (
    SparkSession.builder
    .appName('velox-performance')
    .master(MASTER)
    .config('spark.executor.memory',            '2g')
    .config('spark.driver.memory',              '1g')
    .config('spark.executor.cores',             '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = "GLUTEN/VELOX" if GLUTEN_ENABLED else "VANILLA JVM"
print(f"Spark {spark.version} | Mode: {mode}")
print()
print("IMPORTANT: Run this notebook twice:")
print("  1. make up        → vanilla JVM baseline")
print("  2. make up-gluten → Gluten/Velox results")
print("Compare the printed timings between the two runs.")

## Step 1 — Generate Benchmark Dataset

We write a realistic Parquet dataset to disk.
All benchmarks read from Parquet — this is the most realistic scenario
because it exercises both the Parquet reader and the compute engine.


In [ ]:
random.seed(42)
BENCH_PATH = f"{DATA_DIR}/velox_bench"
pathlib.Path(BENCH_PATH).mkdir(parents=True, exist_ok=True)

N = 1_000_000  # 1M rows — large enough to see real differences

CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture","Toys","Beauty"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR","IN","MX","ES","IT"]
STATUSES   = ["active","inactive","vip","trial","suspended"]
PRODUCTS   = [f"SKU_{i:05d}" for i in range(1, 5001)]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product_sku", StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("status",      StringType()),
    StructField("quantity",    IntegerType()),
    StructField("unit_price",  DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("score",       FloatType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
    StructField("order_ts",    TimestampType()),
])

base_date = datetime.date(2022, 1, 1)
rows = []
for i in range(1, N + 1):
    qty      = random.randint(1, 20)
    price    = round(random.uniform(5, 2000), 2)
    disc     = round(random.uniform(0, 0.4), 3)
    rev      = round(qty * price * (1 - disc), 2)
    odate    = base_date + datetime.timedelta(days=random.randint(0, 730))
    rows.append((
        i,
        random.randint(1, 100_000),
        random.choice(PRODUCTS),
        random.choice(CATEGORIES),
        random.choice(COUNTRIES),
        random.choice(STATUSES),
        qty, price, disc, rev,
        round(random.uniform(1.0, 10.0), 2),
        random.random() < 0.05,
        odate,
        datetime.datetime(odate.year, odate.month, odate.day,
                          random.randint(0,23), random.randint(0,59))
    ))

df = spark.createDataFrame(rows, schema)
df.write.mode("overwrite").parquet(f"{BENCH_PATH}/orders")

row_count = spark.read.parquet(f"{BENCH_PATH}/orders").count()
print(f"Dataset written: {row_count:,} rows → {BENCH_PATH}/orders")
print()

import subprocess
result = subprocess.run(["du","-sh", f"{BENCH_PATH}/orders"], capture_output=True, text=True)
print(f"Parquet size: {result.stdout.split()[0]}")

## Step 2 — Benchmark Framework

We run each query 5 times and take the median to reduce noise.
Results are saved to JSON so you can compare vanilla vs Gluten runs.


In [ ]:
import json as pyjson

RESULTS_PATH = f"{DATA_DIR}/velox_results"
pathlib.Path(RESULTS_PATH).mkdir(parents=True, exist_ok=True)

def bench(name, func, runs=5):
    """Run func() `runs` times, return median time."""
    times = []
    for r in range(runs):
        # Re-read from Parquet each time to avoid caching effects
        t0 = time.time()
        func()
        times.append(round(time.time() - t0, 3))
    times.sort()
    median = times[len(times) // 2]
    p25    = times[len(times) // 4]
    p75    = times[3 * len(times) // 4]
    print(f"  {name:<45} median={median:.3f}s  p25={p25:.3f}s  p75={p75:.3f}s")
    return {"name": name, "mode": mode, "median": median, "p25": p25, "p75": p75, "all": times}

results = []
print(f"Running benchmarks in {mode} mode...")
print(f"{'Query':<47} {'Median':>10}  {'p25':>8}  {'p75':>8}")
print("-" * 78)

## Step 3 — Scan and Filter Benchmarks

These test the Parquet reader + filter evaluation.
Velox's native Parquet reader is one of its biggest advantages —
it avoids JVM deserialization entirely.


In [ ]:
orders = spark.read.parquet(f"{BENCH_PATH}/orders")

# Q1: Full scan + count
r = bench("Q1 full_scan count(*)",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders").count())
results.append(r)

# Q2: Selective filter (high selectivity — few rows pass)
r = bench("Q2 filter: status=vip AND revenue>1000",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .filter((F.col("status") == "vip") & (F.col("revenue") > 1000))
                 .count())
results.append(r)

# Q3: Range scan on numeric column
r = bench("Q3 range: unit_price BETWEEN 100 AND 500",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .filter(F.col("unit_price").between(100, 500))
                 .count())
results.append(r)

# Q4: String filter
r = bench("Q4 string: category IN (Electronics, Books)",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .filter(F.col("category").isin("Electronics", "Books"))
                 .count())
results.append(r)

# Q5: Multi-column projection + filter
r = bench("Q5 project 3 cols + compound filter",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .filter((F.col("is_returned") == False) & (F.col("discount") > 0.2))
                 .select("order_id", "revenue", "country")
                 .count())
results.append(r)

## Step 4 — Aggregation Benchmarks

Aggregations are where Velox shines most.
Vectorized hash aggregation processes entire columns at once using SIMD.


In [ ]:
# Q6: Simple aggregation
r = bench("Q6 SUM + AVG + COUNT (no groupBy)",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .agg(F.sum("revenue"), F.avg("unit_price"),
                      F.count("*"), F.max("score")).collect())
results.append(r)

# Q7: GroupBy aggregation (low cardinality)
r = bench("Q7 groupBy(category) — 8 groups",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .groupBy("category")
                 .agg(F.sum("revenue"), F.avg("discount"),
                      F.count("*"), F.countDistinct("customer_id"))
                 .collect())
results.append(r)

# Q8: GroupBy aggregation (medium cardinality)
r = bench("Q8 groupBy(country, status) — 60 groups",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .groupBy("country", "status")
                 .agg(F.sum("revenue"), F.avg("unit_price"),
                      F.count("*"), F.max("score"))
                 .collect())
results.append(r)

# Q9: GroupBy aggregation (high cardinality)
r = bench("Q9 groupBy(customer_id) — 100K groups",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .groupBy("customer_id")
                 .agg(F.sum("revenue"), F.count("*"),
                      F.avg("discount"), F.max("unit_price"))
                 .collect())
results.append(r)

# Q10: Complex expression in aggregation
r = bench("Q10 complex expr: revenue*(1-discount)*quantity",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .groupBy("category")
                 .agg(F.sum(F.col("revenue") * (1 - F.col("discount")) * F.col("quantity"))
                       .alias("weighted_revenue"),
                      F.avg(F.when(F.col("is_returned"), 0).otherwise(F.col("revenue")))
                       .alias("net_avg_revenue"))
                 .collect())
results.append(r)

## Step 5 — Sort Benchmarks

Sorting requires full materialization of data — Velox uses a native sort
that avoids JVM object overhead.


In [ ]:
# Q11: Sort single column
r = bench("Q11 orderBy(revenue DESC)",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .orderBy(F.desc("revenue"))
                 .limit(1000)
                 .collect())
results.append(r)

# Q12: Sort multiple columns
r = bench("Q12 orderBy(country, category, revenue DESC)",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .orderBy("country", "category", F.desc("revenue"))
                 .limit(1000)
                 .collect())
results.append(r)

# Q13: TopN (sort + limit — optimized path)
r = bench("Q13 top 10 by revenue per category",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .groupBy("category")
                 .agg(F.sum("revenue").alias("total"))
                 .orderBy(F.desc("total"))
                 .limit(10)
                 .collect())
results.append(r)

## Step 6 — Join Benchmarks

Joins require shuffling data across executors. Velox accelerates the
hash table build and probe phases.


In [ ]:
# Create a small dimension table for broadcast join
dim_data = [(i, f"Category_{i}", round(random.uniform(0.05, 0.25), 3))
            for i in range(1, 9)]
dim_df = spark.createDataFrame(dim_data, ["cat_id","cat_name","tax_rate"])
dim_df.write.mode("overwrite").parquet(f"{BENCH_PATH}/categories")

# Q14: Broadcast hash join (small × large)
r = bench("Q14 BroadcastHashJoin (small dim × 1M fact)",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .join(F.broadcast(spark.read.parquet(f"{BENCH_PATH}/categories")),
                       spark.read.parquet(f"{BENCH_PATH}/orders").category ==
                       spark.read.parquet(f"{BENCH_PATH}/categories").cat_name,
                       "left")
                 .agg(F.sum("revenue")).collect())
results.append(r)

# Q15: Self-join aggregation (medium × medium after filter)
r = bench("Q15 self-join: top customers vs avg",
    lambda: (lambda o=spark.read.parquet(f"{BENCH_PATH}/orders"):
             o.groupBy("customer_id").agg(F.sum("revenue").alias("cust_total"))
              .join(o.groupBy("country").agg(F.avg("revenue").alias("country_avg")),
                    how="cross")
              .filter(F.col("cust_total") > F.col("country_avg"))
              .count())())
results.append(r)

## Step 7 — String Operation Benchmarks

String processing is CPU-intensive. Velox uses native string functions
which avoid Java String object overhead and garbage collection.


In [ ]:
# Q16: String operations
r = bench("Q16 UPPER + CONCAT + LENGTH on strings",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .select(
                     F.upper("category").alias("cat_upper"),
                     F.concat("country", F.lit("_"), "status").alias("seg"),
                     F.length("product_sku").alias("sku_len")
                 )
                 .groupBy("cat_upper").count().collect())
results.append(r)

# Q17: LIKE / regex filter
r = bench("Q17 LIKE filter on product_sku",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .filter(F.col("product_sku").like("SKU_001%"))
                 .count())
results.append(r)

# Q18: Date extraction
r = bench("Q18 YEAR + MONTH + DAY extraction from timestamp",
    lambda: spark.read.parquet(f"{BENCH_PATH}/orders")
                 .select(
                     F.year("order_ts").alias("yr"),
                     F.month("order_ts").alias("mo"),
                     F.dayofweek("order_ts").alias("dow")
                 )
                 .groupBy("yr","mo").count().collect())
results.append(r)

## Step 8 — Save Results and Print Report


In [ ]:
# Save results
out_file = f"{RESULTS_PATH}/results_{mode.replace('/', '_').replace(' ', '_')}.json"
pathlib.Path(out_file).write_text(pyjson.dumps(results, indent=2))
print(f"Results saved → {out_file}")
print()

# Print summary
print("=" * 78)
print(f"BENCHMARK SUMMARY — {mode}")
print("=" * 78)
print(f"  Dataset: {N:,} rows | Parquet/zstd | Spark {spark.version}")
print()
print(f"  {'Query':<45} {'Median (s)':>12}  {'p25':>8}  {'p75':>8}")
print("  " + "-" * 76)
for r in results:
    print(f"  {r['name']:<45} {r['median']:>10.3f}s  {r['p25']:>6.3f}s  {r['p75']:>6.3f}s")

## Step 9 — Compare Vanilla vs Gluten Results

Run this cell AFTER you have results from BOTH modes:
1. `make up` → run all cells → results saved
2. `make up-gluten` → run all cells → results saved
3. Restart kernel and run ONLY this cell


In [ ]:
import pathlib, json as pyjson
from pyspark.sql import functions as F

RESULTS_PATH = f"{DATA_DIR}/velox_results"

vanilla_file = pathlib.Path(f"{RESULTS_PATH}/results_VANILLA_JVM.json")
gluten_file  = pathlib.Path(f"{RESULTS_PATH}/results_GLUTEN_VELOX.json")

if not vanilla_file.exists() or not gluten_file.exists():
    missing = [f for f in [vanilla_file, gluten_file] if not f.exists()]
    print(f"Missing result files: {missing}")
    print("Run the notebook in both vanilla and gluten modes first.")
else:
    vanilla = {r["name"]: r["median"] for r in pyjson.loads(vanilla_file.read_text())}
    gluten  = {r["name"]: r["median"] for r in pyjson.loads(gluten_file.read_text())}

    common = [q for q in vanilla if q in gluten]

    print("=" * 78)
    print("VANILLA JVM  vs  GLUTEN/VELOX — Speedup Report")
    print("=" * 78)
    print(f"  {'Query':<45} {'Vanilla':>9}  {'Gluten':>9}  {'Speedup':>9}")
    print("  " + "-" * 76)

    speedups = []
    for q in common:
        v, g   = vanilla[q], gluten[q]
        speedup = round(v / g, 2)
        speedups.append(speedup)
        marker = " ✓" if speedup >= 1.5 else (" =" if speedup >= 0.9 else " ✗")
        print(f"  {q:<45} {v:>7.3f}s  {g:>7.3f}s  {speedup:>7.2f}x{marker}")

    print()
    avg_speedup = sum(speedups) / len(speedups)
    max_speedup = max(speedups)
    max_q       = common[speedups.index(max_speedup)]
    print(f"  Average speedup : {avg_speedup:.2f}x")
    print(f"  Max speedup     : {max_speedup:.2f}x  ({max_q})")
    print(f"  Queries faster  : {sum(1 for s in speedups if s >= 1.2)} / {len(speedups)}")

## Summary: When Does Velox Help Most?

### Operations with highest speedup
- **Aggregations on numeric columns** — vectorized SIMD adds/multiplies entire columns
- **Parquet scan with column pruning** — native reader skips columns at byte level
- **Filter on numeric ranges** — SIMD comparison of entire column batches
- **Hash aggregation** (groupBy + agg) — native hash table, no JVM GC

### Operations with moderate speedup
- **String operations** — faster than JVM but UTF-8 handling adds overhead
- **Sort** — native sort is faster but I/O bound for large datasets
- **Broadcast hash join** — hash table build/probe is faster in native

### When Velox does NOT help
- **Python UDFs** — always fall back to JVM (see notebook 01)
- **Window functions with large shuffle** — serializer incompatibility in 1.6.0
- **Very small datasets** — JVM overhead is minimal, Velox overhead dominates
- **I/O bound workloads** — if bottleneck is disk/network, CPU speedup is invisible

### Practical guidance
> Run the benchmark with **your actual production queries** on representative data.
> Theoretical speedups vary widely depending on data types, cardinality, and query shape.
> Always measure — do not assume.
